In [1]:
%load_ext autoreload
%autoreload 2

import pickle
import numpy as np
import matplotlib.pyplot as plt

import jax
import jax.numpy as jnp
import jax.random as jrand

from abm_event import sample_event, conditional_sample
from abm_ll import city_ll, eval_ds
from stat_utils import CityParams, EventOutcome, GeneralParams, logrange
from city_data import city_data_base, subsampled_city_data, subsample_weights


In [30]:
param_config = {
    "propensity_range": {"range": (-9, -5), "logscale": True},
    "walk_radius_range": {"range": (1, 95), "logscale": False},
    "arm_bias_range": {"range": (-3, 1), "logscale": True},
    "arm_weight_range": {"range": (-1, 1), "logscale": True},
    "atrisk_gather_rate_range": {"range": (-8, -4), "logscale": True}
}

class ParamDistr:
    def __init__(self, param_config):
        self.param_config = param_config
        self.range_lower = jnp.array([config["range"][0] for config in param_config.values()])
        self.range_upper = jnp.array([config["range"][1] for config in param_config.values()])
        self.range_width = self.range_upper - self.range_lower
        self.logscale = jnp.array([config["logscale"] for config in param_config.values()])
        self.param_names = list(param_config.keys())
        self.param_count = len(param_config)

    def sample_range(self, key):
        return jrand.uniform(key, shape=(self.param_count,)) * (self.range_upper - self.range_lower) + self.range_lower
    
    def transform_sample(self, sample):
        return jnp.where(self.logscale, jnp.exp(sample), sample)
    def inverse_transform_sample(self, sample):
        return jnp.where(self.logscale, jnp.log(sample), sample)
    def get_gaussian_proposal(self):
        return lambda key, params : params + jrand.normal(key, shape=(self.param_count,))*self.range_width*0.05
    
    def get_log_gaussian_proposal_pdf(self):
        def log_gaussian_proposal_pdf(p1, p2):
            return -jnp.sum((p1 - p2)**2 / (2 * (0.05 * self.range_width)**2))
        return log_gaussian_proposal_pdf
    
    def get_uniform_prior(self):
        def uniform_prior(p):
            return jnp.all((p >= self.range_lower) & (p <= self.range_upper))+1e-3
        return uniform_prior
    
def eval_params(rngkey, params, param_distr, num_samples, data_dict=None):
    params = param_distr.transform_sample(params)
    params = GeneralParams(*params)
    city_eval = eval_ds(subsampled_city_data, params, 50, rngkey, num_samples=num_samples)
    if data_dict is not None:
        data_dict[params] = city_eval
    total_eval = (subsample_weights.reshape((-1, 1))*city_eval).sum(axis=0)[0]
    return total_eval

param_distr = ParamDistr(param_config)
# params = param_distr.sample_range(jrand.PRNGKey(1))
params = GeneralParams(propensity=2e-09, walk_radius=40, arm_bias=0.01, arm_weight=1, atrisk_gathering_rate=1.02e-05)
params = jnp.array([params.propensity, params.walk_radius, params.arm_bias, params.arm_weight, params.atrisk_gathering_rate])
params = param_distr.inverse_transform_sample(params)
# eval_ll = eval_params(jrand.PRNGKey(1), params, param_distr, 10000)

In [ ]:
def get_param_density_func(param_prior, rngkey, num_samples, data_dict=None):
    def param_density_func(params):
        nonlocal rngkey
        prior = param_prior(params)
        if prior < 0:
            return -1e10
        rngkey, subkey = jrand.split(rngkey)
        params = param_distr.transform_sample(params)
        params = GeneralParams(*np.array(params))
        city_eval = eval_ds(subsampled_city_data, params, 50, subkey, num_samples=num_samples)
        # print(params)
        if( data_dict is not None):
            data_dict[params] = city_eval
        total_eval = (subsample_weights.reshape((-1, 1))*city_eval).sum(axis=0)[0]
        return total_eval + prior
    return param_density_func

P = get_param_density_func(param_distr.get_uniform_prior(), jrand.PRNGKey(1), int(1e4), data_dict={})
proposal_func = param_distr.get_gaussian_proposal()
Q = param_distr.get_log_gaussian_proposal_pdf()

In [45]:
from mcmc import mcmc, mc_update #, gaussian_proposal, gaussian_proposal_pdf
samples, sample_eval = mcmc(params, proposal_func, P, Q, 
                            rng=jrand.PRNGKey(0), num_iter=int(1e2), log=True)

GeneralParams(propensity=1.9999995e-09, walk_radius=40.0, arm_bias=0.01, arm_weight=1.0, atrisk_gathering_rate=1.01999985e-05)
GeneralParams(propensity=2.1746465e-09, walk_radius=35.029545, arm_bias=0.011485768, arm_weight=1.0380658, atrisk_gathering_rate=8.959347e-06)
Accepted: [-20.030119   40.         -4.6051702   0.        -11.493123 ], P=-1352.5198974609375
GeneralParams(propensity=1.954948e-09, walk_radius=41.781967, arm_bias=0.011279838, arm_weight=1.0545423, atrisk_gathering_rate=1.4031151e-05)
Accepted: [-20.052902    41.781967    -4.4847383    0.05310681 -11.174231  ], P=-1306.16552734375
GeneralParams(propensity=1.8146162e-09, walk_radius=35.672573, arm_bias=0.01080667, arm_weight=1.0780221, atrisk_gathering_rate=1.5596854e-05)
Accepted: [-20.052902    41.781967    -4.4847383    0.05310681 -11.174231  ], P=-1306.16552734375
GeneralParams(propensity=1.9976356e-09, walk_radius=38.058327, arm_bias=0.011909851, arm_weight=1.026731, atrisk_gathering_rate=1.6046022e-05)
Accepted: 

KeyboardInterrupt: 

In [38]:
data_dict

NameError: name 'data_dict' is not defined

In [35]:
print(samples, sample_eval)

[-20.03011894  40.          -4.60517025   0.         -11.49312305
 -20.05290222  41.78196716  -4.48473835   0.05310681 -11.17423058
 -20.05290222  41.78196716  -4.48473835   0.05310681 -11.17423058] [-1357.02746582 -1308.08642578 -1308.08642578]


In [15]:
from mcmc import mcmc, mc_update, gaussian_proposal, gaussian_proposal_pdf

In [ ]:
import scipy
def bump_f(x):
    return np.exp(-x**2) + np.exp(-(x-2)**2 * 1/2)
bump_area = np.trapz(bump_f(np.linspace(-10, 10, 100)), np.linspace(-10, 10, 100))
# plt.plot(np.linspace(-10, 10, 100), bump_f(np.linspace(-10, 10, 100))/bump_area)
params = -0.9
samples, sample_eval = mcmc(params, gaussian_proposal, lambda x: np.log(bump_f(x)), 
                            lambda x, y: np.log(gaussian_proposal_pdf(x, y)), 
                            rng=np.random.default_rng(), num_iter=int(1e4), log=True)

In [ ]:

# samples, sample_eval = mcmc(params, gaussian_proposal, bump_f, 
#                             gaussian_proposal_pdf, 
#                             rng=np.random.default_rng(), num_iter=int(4e5), log=False)

In [17]:
plt.figure()
xs = np.linspace(-5, 5, 100)
plt.plot(xs, bump_f(xs)/bump_area)
plt.hist(samples[100:], bins=50, density=True)
plt.show()

NameError: name 'bump_f' is not defined

<Figure size 640x480 with 0 Axes>